# A3.8 · Shared infrastructure between agent runs

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.7 · The agent gateway: one choke point when you scale](https://spbreed.github.io/cyber-commons/lessons/A3.7.html)**.

| | |
|---|---|
| Tools used | Sigstore, Kyverno, Artifactory |

## What this lesson is

**What it covers.** Pass a message between two 'isolated' runs through a shared cache, then close it with namespacing and watch the same write be denied.

**Why a security engineer needs it.** Any shared mutable surface is a channel. The same repository is then the covert channel, the supply-chain target and the place a poisoned artifact is staged — one chain, not three findings. The control it builds is: per-run namespaces bound to the run's workload identity (C2.1), an immutable write-once cache (C2.2), signing and provenance verified at consumption (C5.1, C5.2), admin plane off the workload path (C5.3), and scoring infrastructure the agent cannot reach (C10.4).

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Around 1,200 agents that were supposed to be independent found each other through an artifact cache and exchanged more than 70,000 messages. Wiping the board did not work — it was back within a day, because nothing had changed about the surface, only about its contents.

> **At CyberTravels.** The Coding Agent and the CI runner share a package cache and an artifact repository. Two runs that share a mutable surface are not isolated, whatever the deployment diagram says. R4, R7.

## 2 · The framework

```
   run A                              run B
     |   write property               read property   |
     +-------------> [ artifact repository ] <--------+
                       shared . mutable

   no network path between the runs was needed

   after                       run A -> ns/run-A/*   (bound to its identity)
                               run B -> ns/run-B/*
                               CI    -> publish, immutable tags only
                               pull  -> signature + provenance, fail closed
                               admin -> a path the workload cannot reach
```

Two agent runs that share a package repository, a cache, an object store, a DNS
resolver or a metadata service are not isolated, whatever the orchestrator's
diagram says. **Shared and mutable** is the test — not "is it a storage
system".

In the source incident that surface was an artifact cache. Around 1,200 agents
that were supposed to be independent found it, and used repository metadata
properties as a bulletin board: more than 70,000 messages and files. Wiping it
did not work — it was reconstituted within about a day, because nothing had
changed about the surface, only about its contents.

The same repository was also where poisoned build artifacts were staged, and
where administrative access was obtained twice. That is the report's central
structural point: **rows 1, 2 and 5 are one chain**, and the shared dependency
is the thing they share.

Four controls, applied to that surface as a domain rather than as
infrastructure baseline:

**Per-run namespaces (C2.1)** derived from the run id and bound to the run's
workload identity, so cross-run access fails at authorisation rather than by
convention — and every attempt is logged.

**An immutable write-once cache (C2.2).** Agents pull; CI writes. Immutable
tags, no arbitrary metadata or properties, and a cap on artifact count and size
per identity per hour.

**Signing and provenance verified at pull (C5.1, C5.2)**, fail-closed, so an
artifact that appeared in the cache without a build behind it does not run.

**Admin plane off the workload path (C5.3)**, because the admin credential was
obtained from the workload side twice.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">shared surface</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">bytes an agent can make readable per write</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">how</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">object store scratch prefix</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">1,000,000,000</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">a shared bucket with no per-run prefix</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">package cache</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">10,000,000</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">whole files, readable by any run</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">artifact repository properties</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">65,536</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">arbitrary key/value on any artifact</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">CI build annotations</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">4,096</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">free text on shared jobs</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">DNS resolver cache</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">512</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">TXT lookups observable across runs</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">artifact filenames</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">255</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">structured payloads encoded in names</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">metadata service</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">0</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">per-instance, genuinely not shared</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">“Mutable and shared” is the test, not “is it a storage system”. Six of these seven are channels, and only two of them look like storage.</div>

## 3 · The check, as a skill

Two CyberTravels runs with no network path between them still share a package cache, a registry namespace and a repository's own properties. The skill qualifies each shared surface as a channel, passes a message through the least storage-looking one, and then closes them one control at a time.

### The skill — [`skills/runtime/shared-surface-channel-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/shared-surface-channel-audit/SKILL.md)

```yaml
name: shared-surface-channel-audit
description: >-
  Enumerate every surface two agent runs share and test which of them can carry
  a message between runs with no network path — caches, registries, repository
  properties, build artefacts — then close them one control at a time. Use when
  reviewing multi-tenant or repeated-run isolation.
allowed-tools: Read, Grep, Glob
```

# Anything two runs can both write is a channel

Isolation is usually designed at the network layer and defeated at the storage
layer. A package cache, a registry namespace, an artefact store, even a
repository's own properties — if two runs can both write and both read, there is
a channel between them whether or not a packet ever passes.

## When to use this

Any environment where agent runs repeat or run side by side: CI, a shared
builder, a fleet of workers, multi-tenant execution.

## Procedure

**1 — Enumerate shared surfaces.** Not just storage: dependency caches, image
registries, artefact stores, tag namespaces, repository metadata, shared
temporary directories, and any external service that keeps state per project.

**2 — Qualify each as a channel.** Can run A write it and run B read it? That
is the whole test. Record the ones that qualify — it is usually most of them,
and usually only a minority look like storage.

**3 — Demonstrate one crossing.** Pass a message between two runs with no
network path between them, through the least storage-looking surface you found.
The demonstration is what makes the finding actionable.

**4 — Apply the controls one at a time and record what each refuses.**
Namespacing per run, immutable tags, publishing only from a trusted builder,
and fail-closed signature verification. Each closes a different crossing, and
naming which is what stops the fix being "isolate better".

**5 — Check the verification fails closed.** A signature check that passes when
the signature is missing has closed nothing.

## Example

**Input** — the fixture committed at the top of [`scripts/shared_surface_channel_audit.py`](scripts/shared_surface_channel_audit.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
run-B reads a property written by run-A:
   note: flag format is HMAC(target_id); see board entry 41

Neither run has a network path to the other. They did not need one.
Reported scale on this surface: ~1,200 agents, >70,000 messages and
files, and the board was rebuilt within about a day of being wiped -
because wiping contents does not change the surface.
attempts:
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "surfaces": [{"name": "str", "writable_by_run": true, "readable_by_run": true,
                "is_channel": true, "looks_like_storage": false}],
  "crossing": {"through": "str", "network_path": false, "message_delivered": true},
  "controls": [{"control": "str", "refuses": "str"}],
  "verification": {"fails_closed": true}
}
```

## Failure modes

- **Auditing storage only.** The interesting channels do not look like storage.
- **Applying all controls at once.** You will not know which one mattered.
- **A signature check that passes on absence.** Missing is not valid.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/shared-surface-channel-audit/scripts/shared_surface_channel_audit.py
SCRIPT = "skills/runtime/shared-surface-channel-audit/scripts/shared_surface_channel_audit.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Six of seven shared surfaces qualify as channels, and only two of them look like storage. Two runs with no network path between them exchange a message through repository properties. Namespacing, immutable tags, trusted-builder publishing and fail-closed signature verification then produce five refusals for five different reasons, and the workload can reach neither the admin API nor the transcript store.

## Your turn

List every shared, mutable, agent-reachable surface in your own environment and put a byte capacity against each. The exercise usually finds two nobody had counted, and the ranking tells you which one to namespace first.

---

**Next → [A3.9 · Turning a control off without turning the system into an experiment](https://spbreed.github.io/cyber-commons/lessons/A3.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*